WhatNet - EMNIST-letters

In [ ]:
#  EMNIST Letters: 145,600 chars, 26 balanced classes (A-Z)
#  ~5,600 samples/class. Case-merged: 'A' and 'a' share one label.
#  NOTE: tfds labels are 1-indexed (1=A … 26=Z); remapped to 0-indexed below.
#  NO horizontal flip — b/d and p/q confusions are real in case-merged data.

In [ ]:
# importing necessary
import os, random, json
import numpy as np

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

DATASET      = "emnist/letters"
NUM_CLASSES  = 26
IMG          = 28
BS           = 64
EPOCHS       = 100
LR           = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1        # 0.1 — was hurting hard classes
WARMUP_EP    = 4
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/letters"
AUTOTUNE     = tf.data.AUTOTUNE
LABELS = [chr(c) for c in range(65, 91)]   # ['A', 'B', ..., 'Z']
os.makedirs(RESULTS_DIR, exist_ok=True)

#  DATA LOADING
print(f"[INFO] Loading {DATASET} via tensorflow_datasets ...")
train_raw, info = tfds.load(
    DATASET, split="train", as_supervised=True, with_info=True, shuffle_files=True,
)
test_raw = tfds.load(DATASET, split="test", as_supervised=True)

total   = info.splits["train"].num_examples
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {info.splits['test'].num_examples:,}")

#  PREPROCESSING  —  remap 1-indexed labels → 0-indexed
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    label = tf.cast(label, tf.int32) - 1          # 1-26  ->  0-25
    return image, label

def augment(image, label):
    image = tf.image.random_brightness(image, 0.25)
    image = tf.image.random_contrast(image, 0.75, 1.25)
    image = tf.pad(image, [[3, 3], [3, 3], [0, 0]], constant_values=-1.0)
    image = tf.image.random_crop(image, [IMG, IMG, 1])
    # NO horizontal flip — b/d and p/q confusions are real
    image = image + tf.random.normal(tf.shape(image), stddev=0.03)
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

def to_onehot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

# Fixed -> shuffle the full dataset before splitting to avoid biased val set
train_raw = train_raw.shuffle(total, seed=SEED, reshuffle_each_iteration=False)
val_raw   = train_raw.take(n_val)
train_raw = train_raw.skip(n_val)

train_ds = (
    train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
val_ds = (
    val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
test_ds = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
test_ds_oh = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

steps_per_epoch = n_train // BS
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

#  BUILDING BLOCKS
def gelu(x): return tf.nn.gelu(x)

def channel_attention(x, channels, reduction=4):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(max(1, channels // reduction), activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def residual_block(x, channels):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, r]); x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_ch, out_ch):
    if in_ch != out_ch:
        x = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    r1  = residual_block(x,  out_ch)
    r2  = residual_block(r1, out_ch)
    r3  = residual_block(r2, out_ch)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    return out

def adaptive_filter_capsule(x, num_classes, capsule_dim=32):
    """
    FIX: replaced broken slice `t[:, :, :capsule_dim]` on the 512-dim fused
    vector with a proper Dense projection to capsule_dim.  The old code only
    used the first 32 of 512 features and caused zero-TP for classes 20-25.
    """
    # Capsule prediction matrix
    h = layers.Dense(256, activation=gelu)(x)
    h = layers.Dense(num_classes * capsule_dim)(h)
    h = layers.Reshape((num_classes, capsule_dim))(h)

    # FIX: project x to capsule_dim via Dense, not slicing
    x_proj = layers.Dense(capsule_dim, use_bias=False)(x)          # (B, capsule_dim)
    x_exp  = layers.RepeatVector(num_classes)(x_proj)              # (B, num_classes, capsule_dim)

    caps = layers.Multiply()([x_exp, h])
    caps = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)  # (B, num_classes)
    return layers.BatchNormalization()(caps)

#  MODEL
def build_model(num_classes=26, image_size=28):
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    #  Stem: 3-path (texture + horizontal + vertical)
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t); t = layers.Activation(gelu)(t)
    h = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h); h = layers.Activation(gelu)(h)
    v = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v); v = layers.Activation(gelu)(v)
    stem = layers.Concatenate()([t, h, v])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem); stem = layers.Activation(gelu)(stem)

    #  Encoder
    enc1 = dense_res_block(stem, 64,  64)
    enc2 = dense_res_block(enc1, 64,  128)
    enc3 = dense_res_block(enc2, 128, 256)

    #  Multi-scale GAP fusion
    gap1 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc1))
    gap2 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc2))
    gap3 = layers.GlobalAveragePooling2D()(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])  # (B, 512)

    #  Adaptive Filter Capsule
    afc_out = adaptive_filter_capsule(fused_gap, K, capsule_dim=32)   # (B, K)

    #  Dense head
    x = layers.Dense(128, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation(gelu, name="head_act")(x)
    x = layers.Dropout(0.30)(x)   # reduced from 0.35
    x = layers.Dense(K, name="head_logits")(x)

    #  Gated fusion: AFC + dense head
    combined   = layers.Concatenate(name="gate_input")([x, afc_out])
    gate       = layers.Dense(2, activation="softmax", name="gate")(combined)
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x, gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc")([afc_out, gate])
    out = layers.Add(name="logits")([x_scaled, afc_scaled])

    return Model(inputs=inp, outputs=out, name="WhatNet_Letters")

#  LR SCHEDULE
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base, total_steps, warmup_steps):
        self.base         = base
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)

    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base * step / tf.maximum(self.warmup_steps, 1.0)
        progress  = (step - self.warmup_steps) / tf.maximum(
            self.total_steps - self.warmup_steps, 1.0)
        cosine_lr = tf.maximum(self.base * 0.5 * (1.0 + tf.cos(np.pi * progress)), 1e-6)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {"base": self.base, "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}

#  TRAIN
model        = build_model(NUM_CLASSES, IMG)
warmup_steps = steps_per_epoch * WARMUP_EP
lr_sch       = WarmupCosineDecay(LR, total_steps, warmup_steps)
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_sch, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy"],
    jit_compile=True,
)
print(f"[INFO] Params: {model.count_params():,}")

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        ModelCheckpoint(os.path.join(RESULTS_DIR, "best_model.keras"),
                        monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=15,
                      restore_best_weights=True, verbose=1),
    ],
)

#  EVALUATE
#  Fixed -> use test_ds (which has preprocess applied → 0-indexed labels) for the
#  manual per-class loop.  The old code used test_ds there too, but the labels
#  coming out of test_ds are already 0-indexed after preprocess, so the loop
#  itself was correct — the real bug was the capsule slicing above.
#  Keeping the loop on test_ds (0-indexed) for clarity and correctness.
test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
test_acc *= 100.0

tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)

for images, labels in test_ds:           # labels are 0-indexed (preprocess applied)
    preds = tf.argmax(model(images, training=False), axis=1).numpy()
    lbls  = labels.numpy()
    for c in range(NUM_CLASSES):
        tp[c] += np.sum((preds == c) & (lbls == c))
        fp[c] += np.sum((preds == c) & (lbls != c))
        fn[c] += np.sum((preds != c) & (lbls == c))
        correct_per_class[c] += np.sum(preds[lbls == c] == c)
        total_per_class[c]   += np.sum(lbls == c)

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {model.count_params():,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "EMNIST_LETTERS", "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4), "params": model.count_params(),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, f, indent=2)

print(f"\n[INFO] Saved to {RESULTS_DIR}/"); print("[DONE]")

In [ ]:
 2026-04-26 09:48:03.516787: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
E0000 00:00:1777196883.748028      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777196883.809070      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777196884.336376      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777196884.336420      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777196884.336424      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777196884.336427      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
[INFO] Loading emnist/letters via tensorflow_datasets ...
WARNING:absl:Variant folder /root/tensorflow_datasets/emnist/letters/3.1.0 has no dataset_info.json
Downloading and preparing dataset Unknown size (download: Unknown size, generated: Unknown size, total: Unknown size) to /root/tensorflow_datasets/emnist/letters/3.1.0...
Dl Completed...: 0 url [00:00, ? url/s]Dl Size...: 0 MiB [00:00, ? MiB/s]Extraction completed...: 0 file [00:00, ? file/s]Extraction completed...: 0 file [00:00, ? file/s]Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]Generating train examples...: 0 examples [00:00, ? examples/s]I0000 00:00:1777196921.654201      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777196921.659996      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Shuffling /root/tensorflow_datasets/emnist/letters/incomplete.HBRPOI_3.1.0/emnist-train.tfrecord*...:   0%|   …Generating test examples...: 0 examples [00:00, ? examples/s]Shuffling /root/tensorflow_datasets/emnist/letters/incomplete.HBRPOI_3.1.0/emnist-test.tfrecord*...:   0%|    …Dataset emnist downloaded and prepared to /root/tensorflow_datasets/emnist/letters/3.1.0. Subsequent calls will reuse this data.
[INFO] Train: 79,920 | Val: 8,880 | Test: 14,800
[INFO] Steps/epoch: 1248
[INFO] Params: 5,512,700
Epoch 1/100
WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1777197008.824159     134 service.cc:152] XLA service 0x466ba0d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777197008.824213     134 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777197008.824217     134 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777197012.717394     134 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777197032.916741     134 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 172s 98ms/step - accuracy: 0.1716 - loss: 3.0374 - val_accuracy: 0.8842 - val_loss: 1.0340
Epoch 2/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 100s 80ms/step - accuracy: 0.8706 - loss: 1.0907 - val_accuracy: 0.9145 - val_loss: 0.8910
Epoch 3/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9107 - loss: 0.9436 - val_accuracy: 0.9229 - val_loss: 0.8664
Epoch 4/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9235 - loss: 0.8929 - val_accuracy: 0.9231 - val_loss: 0.8580
Epoch 5/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.9325 - loss: 0.8552 - val_accuracy: 0.9279 - val_loss: 0.8357
Epoch 6/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 100s 79ms/step - accuracy: 0.9398 - loss: 0.8232 - val_accuracy: 0.9419 - val_loss: 0.7932
Epoch 7/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 100s 79ms/step - accuracy: 0.9441 - loss: 0.7983 - val_accuracy: 0.9444 - val_loss: 0.7855
Epoch 8/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 100s 80ms/step - accuracy: 0.9467 - loss: 0.7813 - val_accuracy: 0.9452 - val_loss: 0.7799
Epoch 9/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 97s 78ms/step - accuracy: 0.9506 - loss: 0.7670 - val_accuracy: 0.9401 - val_loss: 0.7921
Epoch 10/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9535 - loss: 0.7564 - val_accuracy: 0.9481 - val_loss: 0.7716
Epoch 11/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.9563 - loss: 0.7445 - val_accuracy: 0.9514 - val_loss: 0.7580
Epoch 12/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9597 - loss: 0.7362 - val_accuracy: 0.9443 - val_loss: 0.7883
Epoch 13/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9617 - loss: 0.7282 - val_accuracy: 0.9509 - val_loss: 0.7623
Epoch 14/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.9626 - loss: 0.7223 - val_accuracy: 0.9497 - val_loss: 0.7575
Epoch 15/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.9633 - loss: 0.7196 - val_accuracy: 0.9510 - val_loss: 0.7553
Epoch 16/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.9667 - loss: 0.7121 - val_accuracy: 0.9461 - val_loss: 0.7799
Epoch 17/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 100s 80ms/step - accuracy: 0.9661 - loss: 0.7120 - val_accuracy: 0.9536 - val_loss: 0.7519
Epoch 18/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 97s 78ms/step - accuracy: 0.9694 - loss: 0.7035 - val_accuracy: 0.9511 - val_loss: 0.7624
Epoch 19/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9712 - loss: 0.6998 - val_accuracy: 0.9526 - val_loss: 0.7628
Epoch 20/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9712 - loss: 0.6975 - val_accuracy: 0.9544 - val_loss: 0.7488
Epoch 21/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9719 - loss: 0.6970 - val_accuracy: 0.9534 - val_loss: 0.7498
Epoch 22/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 97s 77ms/step - accuracy: 0.9735 - loss: 0.6931 - val_accuracy: 0.9534 - val_loss: 0.7518
Epoch 23/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 97s 78ms/step - accuracy: 0.9748 - loss: 0.6888 - val_accuracy: 0.9508 - val_loss: 0.7559
Epoch 24/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9765 - loss: 0.6853 - val_accuracy: 0.9539 - val_loss: 0.7486
Epoch 25/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.9767 - loss: 0.6842 - val_accuracy: 0.9555 - val_loss: 0.7472
Epoch 26/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9782 - loss: 0.6807 - val_accuracy: 0.9523 - val_loss: 0.7591
Epoch 27/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9797 - loss: 0.6789 - val_accuracy: 0.9520 - val_loss: 0.7590
Epoch 28/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9804 - loss: 0.6773 - val_accuracy: 0.9516 - val_loss: 0.7601
Epoch 29/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9819 - loss: 0.6743 - val_accuracy: 0.9511 - val_loss: 0.7672
Epoch 30/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9840 - loss: 0.6703 - val_accuracy: 0.9514 - val_loss: 0.7571
Epoch 31/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9839 - loss: 0.6695 - val_accuracy: 0.9546 - val_loss: 0.7547
Epoch 32/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9854 - loss: 0.6660 - val_accuracy: 0.9539 - val_loss: 0.7558
Epoch 33/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.9865 - loss: 0.6634 - val_accuracy: 0.9563 - val_loss: 0.7507
Epoch 34/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9877 - loss: 0.6622 - val_accuracy: 0.9490 - val_loss: 0.7769
Epoch 35/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 97s 78ms/step - accuracy: 0.9880 - loss: 0.6604 - val_accuracy: 0.9554 - val_loss: 0.7614
Epoch 36/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.9891 - loss: 0.6581 - val_accuracy: 0.9574 - val_loss: 0.7560
Epoch 37/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9893 - loss: 0.6587 - val_accuracy: 0.9561 - val_loss: 0.7584
Epoch 38/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9903 - loss: 0.6565 - val_accuracy: 0.9539 - val_loss: 0.7658
Epoch 39/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9910 - loss: 0.6557 - val_accuracy: 0.9546 - val_loss: 0.7661
Epoch 40/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9924 - loss: 0.6514 - val_accuracy: 0.9560 - val_loss: 0.7594
Epoch 41/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 97s 78ms/step - accuracy: 0.9926 - loss: 0.6515 - val_accuracy: 0.9532 - val_loss: 0.7637
Epoch 42/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 97s 78ms/step - accuracy: 0.9941 - loss: 0.6485 - val_accuracy: 0.9532 - val_loss: 0.7695
Epoch 43/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9929 - loss: 0.6503 - val_accuracy: 0.9537 - val_loss: 0.7736
Epoch 44/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 97s 78ms/step - accuracy: 0.9943 - loss: 0.6477 - val_accuracy: 0.9537 - val_loss: 0.7587
Epoch 45/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 97s 78ms/step - accuracy: 0.9944 - loss: 0.6461 - val_accuracy: 0.9530 - val_loss: 0.7681
Epoch 46/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9955 - loss: 0.6438 - val_accuracy: 0.9521 - val_loss: 0.7712
Epoch 47/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 78ms/step - accuracy: 0.9948 - loss: 0.6452 - val_accuracy: 0.9542 - val_loss: 0.7729
Epoch 48/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 98s 79ms/step - accuracy: 0.9956 - loss: 0.6429 - val_accuracy: 0.9542 - val_loss: 0.7654
Epoch 49/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.9958 - loss: 0.6425 - val_accuracy: 0.9542 - val_loss: 0.7641
Epoch 50/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 100s 80ms/step - accuracy: 0.9956 - loss: 0.6420 - val_accuracy: 0.9550 - val_loss: 0.7682
Epoch 51/100
1249/1249 ━━━━━━━━━━━━━━━━━━━━ 100s 80ms/step - accuracy: 0.9966 - loss: 0.6403 - val_accuracy: 0.9560 - val_loss: 0.7660
Epoch 51: early stopping
Restoring model weights from the end of the best epoch: 36.

[RESULT] Test Acc : 94.03%
[RESULT] Macro F1 : 94.02%
[RESULT] Test Loss: 0.7925
[RESULT] Params   : 5,512,700

[RESULT] Worst-5 classes:
         'U' (cls 20) = 0.0%
         'Z' (cls 25) = 0.0%
         'Y' (cls 24) = 0.0%
         'X' (cls 23) = 0.0%
         'W' (cls 22) = 0.0%

[INFO] Saved to ./results/letters/
[DONE]